In [ ]:
# Allow importing from src
import sys
sys.path.insert(0, '../src/')

# Fix for draw_geometries crashing on Wayland
import os
os.environ["XDG_SESSION_TYPE"] = "x11"

In [ ]:
import pycolmap as pc
from pathlib import Path
import numpy as np
import torch
from tempfile import TemporaryDirectory
from torchvision.transforms.functional import to_pil_image
import open3d as o3d
import matplotlib.pyplot as plt

from utils.data import load_dtu_data

# Interactive trial of pycolmap

Needs an image and output directory below this one

In [ ]:
images, extrinsics, intrinsics = load_dtu_data("scan97", "../data")

In [ ]:
intrinsics[0]

In [ ]:
db_path = Path("colmap.db").resolve()
img_path = Path("images").resolve()
pc.extract_features(database_path=db_path, image_path=img_path, camera_model="PINHOLE", camera_mode=pc.CameraMode.SINGLE)

In [ ]:
fx, fy, cx, cy = intrinsics[0, 0, 0].item(), intrinsics[0, 1, 1].item(), intrinsics[0, 0, 2].item(), intrinsics[0, 1, 2].item()

cam = pc.Camera(
    camera_id=1,
    model=pc.CameraModelId.PINHOLE,
    width=1600,
    height=1200,
    params=[fx, fy, cx, cy],
)

with pc.Database("colmap.db") as db:
    db.update_camera(cam)

In [ ]:
with pc.Database("colmap.db") as db:
    imgs = db.read_all_images()
    for i, extr in enumerate(extrinsics, start=1):
        for img in imgs:
            if int(img.name.split(".")[0]) == i:
                print(i, img.name)
                db.write_pose_prior(img.image_id, pc.PosePrior(extr[:3, -1].numpy(), pc.PosePriorCoordinateSystem.CARTESIAN))
                break

In [ ]:
pc.match_exhaustive(db_path)

In [ ]:
rec = pc.Reconstruction()

with pc.Database("colmap.db") as db:
    for stuff in db.read_all_cameras():
        rec.add_camera(stuff)

    for stuff in db.read_all_rigs():
        rec.add_rig(stuff)

    for stuff in db.read_all_frames():
        stuff.rig = rec.rig(1)
        img = db.read_image(list(stuff.data_ids)[0].id).name
        img = int(img.split(".")[0])
        w2c = extrinsics[img].numpy()
        w2c[:, 1] *= -1
        w2c[:, 2] *= -1
        w2c = np.linalg.inv(w2c)[:3, :4]
        stuff.set_cam_from_world(1, pc.Rigid3d(w2c))
        rec.add_frame(stuff)

    for stuff in db.read_all_images():
        stuff.frame_id = stuff.image_id
        rec.add_image(stuff)
        rec.register_image(stuff.frame_id)

In [ ]:
pc.triangulate_points(
    rec, "colmap.db", img_path, "output"
)

In [ ]:
c = 0
ls = []
for v in rec.points3D.values():
    if v.error < 0.5 and np.all(np.abs(v.xyz) < 1.0):
        ls.append(np.array(v.xyz, dtype=np.float32))
        c += v.track.length()
arr = np.stack(ls, axis=0)
print(f"Tracked through {c:_} pixels")
arr

In [ ]:
box = o3d.geometry.TriangleMesh.create_box(2, 2, 2)
box = o3d.geometry.LineSet.create_from_triangle_mesh(box)
box.translate([-1, -1, -1])

sfm_cloud = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(arr))
sfm_cloud.paint_uniform_color([0.5, 0.5, 0.5])

o3d.visualization.draw_geometries([
    sfm_cloud,
    box
], lookat=[0,0,0])

# Function esque pycolmap sfm pipeline

In [ ]:
pc.logging.minloglevel = 2

pc_img_to_torch_num = lambda img: int(img.name.split(".")[0])
images, extrinsics, intrinsics = load_dtu_data("scan122", "../data")

with TemporaryDirectory() as temp_dir:
    temp_dir = Path(temp_dir).resolve()

    img_dir = temp_dir / "images"
    img_dir.mkdir()

    # Save images to temp
    for i, image in enumerate(images):
        # Not saving transparency as colmap doesn't handle it anyway, if alpha channel exists it is filtered later
        to_pil_image(image[..., :3].permute(2, 0, 1)).save(img_dir / f"{i:04d}.bmp")
    
    # Feature extraction
    db_path = temp_dir / "colmap.db"
    pc.extract_features(database_path=db_path, image_path=img_dir, camera_model="PINHOLE", camera_mode=pc.CameraMode.SINGLE)

    # Set camera params
    fx, fy, cx, cy = intrinsics[0, 0, 0].item(), intrinsics[0, 1, 1].item(), intrinsics[0, 0, 2].item(), intrinsics[0, 1, 2].item()
    cam = pc.Camera(
        camera_id=1,
        model=pc.CameraModelId.PINHOLE,
        width=1600,
        height=1200,
        params=[fx, fy, cx, cy],
    )
    with pc.Database(db_path) as db:
        db.update_camera(cam)

    # Add pose priors from extrinsics
    with pc.Database(db_path) as db:
        imgs = db.read_all_images()
        for i, extr in enumerate(extrinsics):
            for img in imgs:
                if pc_img_to_torch_num(img) == i:
                    db.write_pose_prior(img.image_id, pc.PosePrior(extr[:3, -1].numpy(), pc.PosePriorCoordinateSystem.CARTESIAN))
                    break

    # Feature matching
    pc.match_exhaustive(db_path)

    # Populate reconstruction params from db (cams, rigs, frames, images)
    rec = pc.Reconstruction()
    with pc.Database(db_path) as db:
        for stuff in db.read_all_cameras():
            rec.add_camera(stuff)

        for stuff in db.read_all_rigs():
            rec.add_rig(stuff)

        for stuff in db.read_all_frames():
            stuff.rig = rec.rig(1)
            img = db.read_image(list(stuff.data_ids)[0].id)
            img = pc_img_to_torch_num(img)
            # Colmap uses y down, z forward, also needs w2c instead of c2w
            c2w = extrinsics[img].numpy()
            c2w[:, 1] *= -1
            c2w[:, 2] *= -1
            w2c = np.linalg.inv(c2w)[:3, :4]
            stuff.set_cam_from_world(1, pc.Rigid3d(w2c))
            rec.add_frame(stuff)

        for stuff in db.read_all_images():
            stuff.frame_id = stuff.image_id
            rec.add_image(stuff)
            rec.register_image(stuff.frame_id)

    # Find sparse reconstruction points
    pc.triangulate_points(
        rec, db_path, img_dir, temp_dir / "output"
    )

In [ ]:
ls = []

has_alpha = images.shape[-1] > 3
depths = torch.full(images.shape[:-1], -1, dtype=torch.float32)
errors = torch.full(images.shape[:-1], torch.inf, dtype=torch.float32)

for img in rec.images.values():
    torch_img_num = pc_img_to_torch_num(img)
    origin = extrinsics[torch_img_num, :3, -1]

    for p2 in img.points2D:
        if p2.point3D_id - 2**64 == -1:  # -1 id (no point3D) is represented in uint as 2**64 - 1
            continue

        p3 = rec.points3D[p2.point3D_id]
        if np.any(np.abs(p3.xyz) > 1.0):  # out of observed bbox
            continue

        x, y = torch.tensor(p2.xy).round().to(torch.int32) - 1
        dist = torch.sqrt(torch.sum(torch.pow(origin - torch.tensor(p3.xyz, dtype=torch.float32), 2), -1))

        # Update to new point with less error and ignore anything with alpha=0
        if errors[torch_img_num, y, x] > p3.error and (not has_alpha or images[torch_img_num, y, x, -1] > 0):
            errors[torch_img_num, y, x] = p3.error
            depths[torch_img_num, y, x] = dist

            ls.append(np.array(p3.xyz, dtype=np.float32))

print((errors != torch.inf).sum())

plt.hist(errors[errors != torch.inf], bins=100)
plt.title("Reprojection error barplot")

box = o3d.geometry.TriangleMesh.create_box(2, 2, 2)
box = o3d.geometry.LineSet.create_from_triangle_mesh(box)
box.translate([-1, -1, -1])

arr = np.stack(ls, axis=0)
sfm_cloud = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(arr))
sfm_cloud.paint_uniform_color([0.5, 0.5, 0.5])

o3d.visualization.draw_geometries([
    sfm_cloud,
    box
], lookat=[0,0,0])